<a href="https://colab.research.google.com/github/csbell-vu/in-class-notebooks/blob/main/transformer_architecture_explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer Architecture Drilldown
> Hands-on exploration of the OPT-125M model and its components

**How to use this notebook**

1. Run the cells in order.
2. When you see **Before you run**, stop and make a prediction. Then run the code and compare what you expected with what you actually see. These predictions are for your own learning and are not submitted.
3. When you reach a **Brightspace question**, answer the numbered question in Brightspace. Only the Brightspace responses are submitted.

Throughout the notebook, keep asking the same concrete questions: What goes into this component? What does the component do? What comes out? What do the dimensions tell you about that calculation?

The goal is to give you actual model weights, activations, and outputs to reason from as you work through the architecture again.

In [ ]:
# Install necessary libraries
!pip install -q transformers accelerate

In [ ]:
import torch
from transformers import GPT2Tokenizer, OPTForCausalLM

## Generation with OPT

The following process generates text using a transformer.

In [ ]:
# Load the smallest OPT model and its corresponding tokenizer
# Note: OPT models use the GPT2Tokenizer class
model_name = "facebook/opt-125m"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = OPTForCausalLM.from_pretrained(model_name, attn_implementation="eager")

# Set the model to evaluation mode (no training)
model.eval()

# Define the input text
input_text = "Hello, I am a language model,"

# Tokenize the input text
inputs = tokenizer(input_text, return_tensors="pt")

# Generate tokens using the model
generate_ids = model.generate(inputs.input_ids, max_length=30)

# Decode and print the output
output_text = tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
print("Generated Output:")
print(output_text)

However, this usage obscures the actual internal behavior of the transformer. In this exploration, we uncover, layer-by-layer, the internal operation of the transformer.

## 1. Model Architectural Choices

Run `model.config` below. These are the architectural decisions for the model we are using. We will return to them throughout the notebook because they determine the dimensions of the embedding tables, attention heads, decoder blocks, and MLP layers.

In [ ]:
model.config

As a warmup, **Complete Question S1-Q1 in Brightspace**. This question will ask you to explain the behavior of the parameters and the consequence for the model. An example of the level of detail/content an answer is provided below:

> **Example answer:** `use_cache = True` enables the **key-value (KV) cache**. During generation, the KV cache saves key and value vectors that have already been calculated so the model can reuse them when it generates the next token.

### Ballpark Model Layers

The printed model is an indented outline. When one component is indented inside another, it belongs to that larger component. Use that nesting to find:

- the **transformer body**, `model.model`;
- the **decoder stack**, `model.model.decoder.layers`;
- one **decoder block**, such as `model.model.decoder.layers[0]`; and
- the **language modeling head**, `model.lm_head`, which sits outside the transformer body.

Within a decoder block, look for both self-attention and the MLP. A block is not the same thing as one attention head, and the 12 blocks in the stack are not the same as the 12 attention heads inside each block.

_One warning: the printed order tells you what modules are registered inside the model, but it does not always show the order in which activations move through them. In this OPT printout, for example, `final_layer_norm` appears before the decoder blocks even though the forward pass applies it after the blocks._

**Before you run:** On a piece of paper, sketch how you expect these parts to appear as an indented list. Then run the cell and compare your outline with the actual nesting.

In [ ]:
# print model layers
model

Based on these observations, answer **Brightspace Question S1-Q2**.

## 2. From Text to Token Representations
### Tokenization

The tokenizer is used before the text enters the transformer. It turns the text into token IDs.

In [ ]:
# Define the input text
input_text = "Hello, I am a language model,"

# Tokenize the input text
inputs = tokenizer(input_text, return_tensors="pt")

# Print total number of tokens:
print(f"> Total Number of Tokens: {len(inputs['input_ids'][0])}\n")
print(f"> Content of the 'inputs' variable:\n")

# Show actual outputs
inputs

The text contains eight words or punctuation marks, but the tokenizer produced nine token IDs. **Why might the number of tokens differ from the number of words?** Give at least two reasons.

<details>
  <summary><font color="blue"><b>Check your reasoning</b></font></summary>
  <p>Two reasons are visible here. The tokenizer can divide text into subword tokens, so one word does not always correspond to one token. OPT also adds a special beginning-of-sequence token, which explains the extra token in this input.</p>
</details>

Now that we have token IDs, the next question is: **How does the model turn each ID into a learned vector representation?**

In [ ]:
# See the tokens produced by the tokenizer
tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

---

# The Transformer Body

## Token Representations

### The Token Embedding Matrix

#### Structure

The token embedding matrix, $W_e$, transforms any token ID into its learned vector representation. An embedding layer does not multiply the token ID by the matrix. It uses the ID to look up one row, so the matrix acts as a learned lookup table.

**Before you run:** $W_e$ must be able to transform any token ID in this model's vocabulary into one vector representation. What dimensions must the table have to do that? What does each dimension count?

In [ ]:
# This is the token embedding weight matrix for OPT. Note that the actual output is truncated for visual reasons.
model.model.decoder.embed_tokens.weight

In [ ]:
# dimensions of the token embedding matrix/lookup table
model.model.decoder.embed_tokens.weight.shape

#### Usage

Our input has nine token IDs, not every token in the vocabulary. The next cell uses those nine IDs to look up nine rows from the embedding matrix.

**Before you run:** How many rows should be returned? How long should each row be? If the input IDs changed, would the stored embedding matrix change, or would the model select different rows from the same matrix?

In [ ]:
# Use the embedding lookup to transform the input IDs
token_embedding_activations = model.model.decoder.embed_tokens(inputs['input_ids'])
token_embedding_activations

In [ ]:
token_embedding_activations.shape

### Weights and Activations

Compare the two tensors you just printed:

- `model.model.decoder.embed_tokens.weight` is the full learned lookup table stored in the model.
- `token_embedding_activations` is the output produced when that table is used for this particular sequence of nine token IDs.

The first tensor exists before a particular input is supplied as a part of the model. These are the **weights** that populate the layers of the model. The second exists because this input was processed, and is called **an activation**.

**Now answer Brightspace questions S2-Q03 through S2-Q05 based on your observations.**

### Position Embeddings

#### Structure

Look back at `max_position_embeddings = 2048`. This is the largest number of token positions configured for this model. The stored position embedding matrix has 2050 rows because OPT retains the positional-index convention used by its Fairseq implementation: position index 1 is used for padding, and ordinary token positions begin at index 2. Preserving that numbering shifts the token positions upward by two, so the implementation allocates two extra rows. Those rows do not extend the context window.

**Before you run:** Predict the dimensions of the stored position embedding **weights**. Does each position vector need the same width as a token embedding? Why or why not?

In [ ]:
model.model.decoder.embed_positions.weight

In [ ]:
model.model.decoder.embed_positions.weight.shape

#### Usage

Now use the position embedding matrix to produce activations for this particular input.

**Before you run:** What dimensions should the position embedding output activations have for a sequence of nine tokens?

In [ ]:
position_embedding_activations = model.model.decoder.embed_positions(inputs['attention_mask'])
position_embedding_activations

In [ ]:
position_embedding_activations.shape

### Combine Token and Position Information

For each token, we now have two vectors of the same length: a token embedding and a position embedding. The model adds the two vectors element by element.

**Before you run:** Predict the dimensions of the result. Are the values in the result stored model weights, or are they activations calculated for this input?

In [ ]:
# Add token and position embedding activations
token_representations = token_embedding_activations + position_embedding_activations
token_representations

In [ ]:
# Size
token_representations.shape

**Answer Brightspace questions S2-Q06 and S2-Q07** based on these observations.

---

## 3. One Decoder Layer

The token representations now enter the decoder blocks. This model has 12 decoder blocks, and they are used one after another. We will first take apart block 0 and calculate its output. Then we will pass that output through blocks 1 through 11.

A decoder block is not the same thing as an attention head. One decoder block contains multi-head self-attention, an MLP, layer norms, and residual connections. The language modeling head is not part of the decoder block. We can see the content of a decoder block below.

In [ ]:
model.model.decoder.layers[0]

### Layer Normalization Before Attention

This OPT model applies layer normalization before the attention calculation. The normalization changes the values, but it does not change the dimensions.

**Before you run:** Should layer normalization change the number of tokens or the number of features for each token?

In [ ]:
# apply layer 0 layer self-attention layer norm
ln_token_representations = model.model.decoder.layers[0].self_attn_layer_norm(token_representations)
ln_token_representations

In [ ]:
# look at shape
ln_token_representations.shape

Based on these observations, answer **Brightspace Questions S3-Q08 and S3-Q09** on your worksheet.

### Self-Attention

Self-attention allows the representation for one token position to use information from other token positions. Because OPT is a causal language model, a position can use information from itself and earlier positions, but it cannot use information from later positions.

We will calculate attention in several small steps using the actual weights from block 0. Begin by looking at the attention module itself.

`q_proj` ($W_q$), `k_proj` ($W_k$), `v_proj` ($W_v$), and `out_proj` ($W_o$) are linear layers (weights + biases) in the trained model. Their `.weight` tensors contain learned weights. When one of these layers operates on an input, its output is an activation.

Be careful with the phrase *attention weights*. Here, `attention_weights` will refer to the post-softmax values calculated for the current input. They are activations, not trained parameters. The learned projection weights are the tensors stored in `q_proj.weight`, `k_proj.weight`, `v_proj.weight`, and `out_proj.weight`.

In [ ]:
model.model.decoder.layers[0].self_attn

<font color="purple"><b>Below, you will actually see the TRANSPOSED dimensions for these linear layers due to the way they are stored. They are backwards from what we see in the "module definitions". In other words, the dimensions you see are <code>(out_dimensions, in_dimensions)</code> rather than the (in_dimensions, out_dimensions) that we're used to seeing.</b></font>

In [ ]:
# look at weight matrix for q - note that you can change the "q" for "k" and "v" to see others.
model.model.decoder.layers[0].self_attn.q_proj.weight

In [ ]:
# Print shapes of the matrices
print(f"Shape of W_q: {model.model.decoder.layers[0].self_attn.q_proj.weight.shape}")
print(f"Shape of W_k: {model.model.decoder.layers[0].self_attn.k_proj.weight.shape}")
print(f"Shape of W_v: {model.model.decoder.layers[0].self_attn.v_proj.weight.shape}")
print(f"Shape of W_out: {model.model.decoder.layers[0].self_attn.out_proj.weight.shape}")


#### Step 1. Calculate Q, K, and V

The normalized token representations have dimensions `(1, 9, 768)`. Three different learned projections operate on the 768 features of every token. Each one produces a new `(1, 9, 768)` activation.

The dimensions match, but the three outputs have different jobs. Queries represent what a token position is looking for. Keys represent what each position can be matched on. Values contain the information that attention may combine.

**Before you run:** Predict the dimensions of the three outputs. Are Q, K, and V learned weights, or are they activations calculated from this input?

In [ ]:
attention = model.model.decoder.layers[0].self_attn

queries = attention.q_proj(ln_token_representations)
keys = attention.k_proj(ln_token_representations)
values = attention.v_proj(ln_token_representations)

print("Queries:", queries.shape)
print("Keys:", keys.shape)
print("Values:", values.shape)

#### Step 2. Divide Q, K, and V Across the 12 Attention Heads

Each 768-feature projection contains 12 heads with 64 features per head. We separate those heads so that each head can calculate its own attention pattern. This reshape does not calculate new values. It only exposes the 12 groups already contained in the last dimension.

In [ ]:
queries = queries.reshape(1, 9, 12, 64).transpose(1, 2)
keys = keys.reshape(1, 9, 12, 64).transpose(1, 2)
values = values.reshape(1, 9, 12, 64).transpose(1, 2)

print("Queries by head:", queries.shape)
print("Keys by head:", keys.shape)
print("Values by head:", values.shape)

#### Step 3. Calculate How Strongly Each Query Matches Each Key

For one head, each of the nine query vectors is multiplied by each of the nine key vectors. The result is a 9 by 9 score matrix. Each row identifies the token position asking the question. Each column identifies a position whose information might be used.

In [ ]:
attention_scores = queries @ keys.transpose(-1, -2)
attention_scores = attention_scores / (64 ** 0.5)

print("Scaled attention scores:", attention_scores.shape)
attention_scores[0, 0]

#### Step 4. Apply the Causal Mask and Softmax

Rows in the mask represent query positions, and columns represent key and value positions. Values above the diagonal are negative infinity because those columns represent future positions. After softmax, those future positions will have probability 0.

Our input has no padding, so we can build this causal mask directly. The first two dimensions let the same 9 by 9 pattern be used for the batch and all 12 heads.

**Before you run:** For query position 3, counting from 0, which key positions should have probability 0?

In [ ]:
causal_mask = torch.triu(
    torch.full(
        (1, 1, 9, 9),
        float("-inf"),
        device=attention_scores.device,
        dtype=attention_scores.dtype,
    ),
    diagonal=1,
)
causal_mask[0, 0]

In [ ]:
masked_scores = attention_scores + causal_mask
attention_weights = torch.softmax(masked_scores, dim=-1)

print("Attention weights:", attention_weights.shape)
print("Row sums for head 0:", attention_weights[0, 0].sum(dim=-1))
attention_weights[0, 0]

#### Step 5. Use the Attention Weights to Combine the Value Vectors

Each row now shows how much one token position will use the value from every position it is allowed to see. Multiplying these attention weights by the value vectors produces one weighted combination for each token and each head.

**Before you run:** For one head, will the result for each token contain 9 features or 64 features? Will this calculation change the number of token positions?

In [ ]:
contextualized_values = attention_weights @ values
print("Weighted values by head:", contextualized_values.shape)

#### Step 6. Put the Head Outputs Back Together

Each token now has 12 results with 64 features each. Put those 12 groups back together so that every token has 768 features again. `out_proj` then applies a learned transformation to those 768 features for each token. It does not mix information between token positions.

**Before you run:** What dimensions must the final attention output have so that it can be added to the residual?

In [ ]:
combined_heads = contextualized_values.transpose(1, 2).reshape(1, 9, 768)
attention_output = attention.out_proj(combined_heads)

print("Combined heads:", combined_heads.shape)
print("Attention output:", attention_output.shape)

**Brightspace questions S3-Q10 through S3-Q14:** Explain what changed at each step: the Q, K, and V projections; the split into heads; the score matrix; the mask and softmax; the weighted values; and the output projection.

### Add the Attention Residual

Attention produced a new `(1, 9, 768)` activation. The block adds it to the token representations that entered the block. It does not add the normalized copy; that copy was used only as the input to attention.

**Before you run:** Why must these two tensors have the same dimensions?

In [ ]:
attention_sublayer_output = token_representations + attention_output
print("After the attention residual:", attention_sublayer_output.shape)

In [ ]:
attention_sublayer_output.shape

**Brightspace question S3-Q15:** Why must `out_proj` return 768 features for each token before the residual can be added?

### The MLP Sublayer

Attention allows information to move among token positions. The MLP does something different: it applies the same learned transformation to the features of each token independently.

`fc1` increases the number of features for each token from 768 to 3072. The activation function operates on those values. Then `fc2` reduces the number of features back to 768.

**Before you run:** If the input dimensions are `(1, 9, 768)`, which dimension should change after `fc1`? Can these MLP operations move information from one token position to another? Why or why not?

In [ ]:
print(f"MLP Layer Specification for fc1: {model.model.decoder.layers[0].fc1}")
print(f"MLP Layer Specification for fc2: {model.model.decoder.layers[0].fc2}\n")

# you can look at the weights yourself if you like. Change fc1 to fc2 as desired.
model.model.decoder.layers[0].fc1.weight

In [ ]:
# apply the FFNN pre-layer norm
mlp_input = model.model.decoder.layers[0].final_layer_norm(attention_sublayer_output)
mlp_input.shape

In [ ]:
# use the first dimension of the FFNN to transform the attention sublayer output
fc1_output = model.model.decoder.layers[0].fc1(mlp_input)
fc1_output.shape

In [ ]:
# apply the activation function for the FFNN
relu_output = model.model.decoder.layers[0].activation_fn(fc1_output)
relu_output.shape

In [ ]:
# Project 3072 features back to the model's 768-feature residual width.
fc2_out = model.model.decoder.layers[0].fc2(relu_output)
fc2_out.shape

In [ ]:
# Add the residual the processed blow
mlp_output = attention_sublayer_output + fc2_out
print('Complete decoder block 0 output:', tuple(mlp_output.shape))

**Use this exploration to answer Brightspace questions S3-Q16 and S3-Q17.**

---

## 4. The Decoder Stack: Iteration and The Rest of the Transformer Body

We have finished block 0. Blocks 1 through 11 still need to operate on the activations, in order. Each block receives the output from the block before it. The dimensions usually remain `(1, 9, 768)`, but the values continue to change.

**Before you run:** Explain why 12 decoder blocks and 12 attention heads describe two different kinds of repetition in this model.

In [ ]:
# Iterate over the rest of the decoder layers/blocks
for layer_idx, layer in enumerate(model.model.decoder.layers):

  if layer_idx == 0:
    # skip it because we just did it above
    continue

  # otherwise, let's treat each decoder block as the operator and operate on each layer
  mlp_output = layer(hidden_states = mlp_output, attention_mask=causal_mask)
  print(f"Finished Layer {layer_idx}: Shape of output of MLP: {mlp_output.shape}")

**Use this exploration to answer Brightspace question S4-Q18.**

### Final Layer Normalization

After the activations have passed through all 12 decoder blocks, the transformer body applies one final layer normalization. The language modeling head comes after the body.

In [ ]:
final_hidden_state = model.model.decoder.final_layer_norm(mlp_output)
final_hidden_state.shape

---

## 5. The Language Modeling Head

The transformer body has produced a 768-feature vector for each of the nine token positions. The language modeling head uses each of these 9 representations to calculate one logit (score) for every token in the vocabulary.

**Before you run:** What output dimensions do you expect? Which configuration setting determines the number of logits calculated for each position?

In [ ]:
# The language modeling head dimensions
model.lm_head

In [ ]:
# operate on the hidden states using the LM head
logit_outputs = model.lm_head(final_hidden_state)
logit_outputs

In [ ]:
# look at the shape
logit_outputs.shape

### Weight Tying: The token embedding matrix and the LM Head

In this OPT model, the token embedding matrix and the language modeling head use the same stored weights. This is called **weight tying**.

The two components still perform different operations. The embedding component uses a token ID to select one row from the matrix. The language modeling head uses the hidden state output from the transformer body to calculate a logit for every vocabulary token. PyTorch stores the shared matrix with dimensions `(vocab_size, hidden_size)` and uses it in the appropriate orientation for the linear calculation.

In [ ]:
embedding_weights = model.model.decoder.embed_tokens.weight
lm_head_weights = model.lm_head.weight
print('Embedding weight shape:', tuple(embedding_weights.shape))
print('LM head weight shape:', tuple(lm_head_weights.shape))
print('Same underlying stored weights:', embedding_weights.data_ptr() == lm_head_weights.data_ptr())

**Using the exploration above, answer Brightspace questions S5-Q19 through S5-Q21.**

### Sampling the Next Token for Generation

The model calculated vocabulary logits for all nine input positions. To choose the next token, we use the logits from the token at the last position in the sequene. The cell below shows the five largest logits. These are scores, not probabilities yet, but we can use them all the same for this purpose.

In [ ]:
top5 = logit_outputs[0, -1, :].topk(5)
top5

In [ ]:
top5.values[0]

In [ ]:
tokenizer.decode(top5.indices[0])

---

## 6. Iterative Generation and the KV Cache

Now, we'll use the complete model to iteratively generate text. Pay attention to how many input tokens are processed on each run and what the model saves in the KV cache.

On the first run, the model processes the entire prompt and saves the calculated keys and values for those token positions. This first calculation is sometimes called **prefill**. On later runs, the model processes the new token and reuses the keys and values already saved for the earlier tokens. This is the **decoding** stage.

The cache contains K and V activations calculated for the input. It does not contain the learned K and V projection matrices. A new query still has to be calculated in every decoder block for each new token.

### Reset the Model Inputs

Run this cell before you begin. If you want to start the generation over later, run it again before continuing.

In [ ]:
# Define the input text
input_text = "Hello, I am a language model,"

# Tokenize the input text
inputs = tokenizer(input_text, return_tensors="pt")

# Parse out our "inputs" into individual elements (useful for next cell)
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
past_key_values = None

### Generate One Token at a Time

Run this cell iteratively. To do this, you can click the arrow next to the cell or click Cmd+Enter (Ctrl+Enter on Windows).

Run the next cell at least twice, making sure to inspect the outputs of the cell as they change. The first run uses all of the prompt tokens. Each later run supplies only the token selected on the previous run, along with the saved KV cache.

**Before you run:** Which tensors should grow in sequence length as tokens are generated? Which values are learned model parameters that remain fixed?

In [ ]:
print("New input_ids supplied to this forward pass:", tuple(input_ids.shape))
model_outputs = model(input_ids=input_ids,
                      attention_mask=attention_mask,
                      past_key_values=past_key_values,
                      use_cache=True)

# 1. Get the new token ID (Greedy decoding via argmax)
new_ind = model_outputs.logits[0, -1, :].argmax()

# 2. Decode and update the text
new_word = tokenizer.decode(new_ind)
input_text += new_word
print(input_text)

# 3. Prepare input_ids for the NEXT run (replacing the current inputs)
input_ids = new_ind.unsqueeze(0).unsqueeze(0)

# 4. Prepare past_key_values for the NEXT run
past_key_values = model_outputs.past_key_values

# 5. Prepare attention_mask for the NEXT run
# Append a single '1' to the mask to account for the new token
new_mask_token = torch.ones((1, 1), dtype=torch.long, device=attention_mask.device)
attention_mask = torch.cat([attention_mask, new_mask_token], dim=-1)

# # 6. Looking at past key values. Uncomment when ready for next section.
# print(f"Number of layers in for past_key_values: {len(past_key_values.layers)}")
# print(f"Shape of past keys for layer 0: {past_key_values.layers[0].keys.shape}")
# print(f"Shape of past values for layer 0: {past_key_values.layers[0].values.shape}")

### Look at What the KV Cache Stores

After you have a good sense of the previous exploration, reset the generation. Then, uncomment the last 3 lines of the previous cell.

This output prints the number of decoder blocks represented in the cache and the dimensions of the cached K and V activations for block 0.

Make sure you observe which dimension grows by one after each newly generated token to better understand the content of the KV-cache.

**Use the exploration above to answer Brightspace questions S6-Q22 and S6-Q23.**

---



# Congratulations!

Great! You're all done. Make sure you answer all the questions on Brightspace and then submit online.

**Optional challenge:** Reset the generation and turn off the cache. Change the loop so that it supplies the complete token history on every run. You may also replace greedy selection with temperature sampling. These changes are optional and are not part of the Brightspace submission.